# One-Dimensional Linear-Quadratic (LQ) Mean-Field Control Benchmark

Reference: `files/reference/LQ_framework.tex` (the full derivation), matching `files/reference/continuous_benchmarks.tex`, Sec. "Linear-Quadratic Validation". Unlike every discrete benchmark in this repo, **the perturbed objective $J^\lambda(\theta)$, its exact policy gradient, and the unperturbed optimal policy are all available in closed form** — there is no Monte Carlo estimation error to separate from optimization error here.

**Model.** State and action spaces are $\mathbb R$. $X_0\sim\mathcal N(\mu_0,\Sigma_0)$; the state law $m_t^\theta=\mathcal N(\mu_t^\theta,\Sigma_t^\theta)$ stays Gaussian and evolves deterministically. The policy is Gaussian feedback $\pi_t^\theta(\cdot\mid x,m)=\mathcal N(\theta_t^1x+\theta_t^2\bar m,\tau^2)$; the transition kernel is $P_t(\cdot\mid x,\alpha,m)=\mathcal N(ax+b\alpha+c\bar m,\sigma^2)$. Running/terminal costs are
$$r_t(x,m,\alpha)=qx^2+r\alpha^2+\kappa\bar m^2, \qquad g(x,m)=q_Tx^2+\kappa_T\bar m^2$$
($\kappa,\kappa_T$ are the reference's own $\gamma,\gamma_T$, renamed here to avoid clashing with this repo's RL discount factor — LQ has no discounting). The perturbation randomizes the *population argument* via a shared affine map $T_t^\lambda(x)=(1+\lambda\zeta_t)x+\lambda\beta_t$, $(\zeta_t,\beta_t)\sim\mathcal N(0,\rho^2)^{\otimes2}$, applied once per $t$ to the whole law.

**theta is genuinely time-indexed**: shape $(T,2)$, $\theta_t=(\theta_t^1,\theta_t^2)$ — unlike every other benchmark's stationary/MLP policy, there is no way to evaluate a trained $\theta$ at a horizon other than the one it was trained for.

**Mean-field coupling.** The default `LQConfig` is chosen so the coupling actually matters: $c$ (population feedback in the transition) is comparable to $a$ (self-transition), and $\kappa,\kappa_T$ are comparable to or larger than $q,q_T$. `tests/test_lq.py` checks this numerically: the Riccati-optimal $\theta^\star$ achieves *more than 2x lower cost*, under this environment's own objective, than the optimal $\theta^\star$ of the same problem with the coupling switched off ($c=\kappa=\kappa_T=0$) — ignoring the mean-field term is a substantially worse policy, not a negligible correction.

**Two training algorithms**, both cost-minimizing (note the sign convention: everything here is a *cost*, matching the reference's own notation, unlike the reward-maximization convention used by the rest of this repo — see `mfc.algorithms.lq`'s module docstring):
- **`exact_gradient`**: gradient descent using the reference's closed-form $O(T)$ adjoint gradient (`LQ.exact_gradient`) directly — no sampling at all.
- **`reinforce`**: the classical-REINFORCE ablation (context.md: "Show missing mean-field term by comparison with reinforce") — Monte Carlo score-function estimator that treats the population mean as exogenous, structurally missing the population-sensitivity term that `exact_gradient` captures exactly.

In [ ]:
import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch

torch.set_default_dtype(torch.float64)
torch.set_default_device("cpu")  # LQ's per-step work is small scalar/(T,B) ops: GPU kernel-launch
                                  # overhead dominates here, same finding as this repo's twostate profiling

from configs.lq import MID
from mfc.environments.lq import LQ, LQConfig
from mfc.plotting import diagnostics as viz
from mfc.plotting.style import apply_style, color_for, new_figure, style_legend
from scripts.train import run_continuous
from scripts.test import load_runs

## Configuration and budget

This notebook demonstrates the **mid** run tier from `configs/lq.py`: one seed, the reference horizon $T=5$, and the full training length. Run `scripts/train.py --env lq --alg exact_gradient --config main` (and `--alg reinforce`) separately for the full main-tier sweep (5 seeds, horizons $T\in\{3,5,10\}$, used by the horizon-scaling cell below when available).

In [ ]:
cfg = MID
env = LQ(device="cpu")  # matches the default device set above and scripts.train.run_continuous's own choice
T = cfg.horizons[0]

print(f"algorithms:  {cfg.algorithms}")
print(f"lambdas:     {cfg.lambdas}")
print(f"T={T}, seeds={cfg.seeds}, lr={cfg.lr}, n_train={cfg.n_train}, B(reinforce)={cfg.B}")
print(f"model: a={env.config.a}, b={env.config.b}, c={env.config.c}, sigma={env.config.sigma}")
print(f"cost:  q={env.config.q}, r={env.config.r}, q_T={env.config.q_T}, kappa={env.config.kappa}, kappa_T={env.config.kappa_T}")
print(f"tau={env.config.tau}, rho={env.config.rho}, mu0={env.config.mu0}, Sigma0={env.config.Sigma0}")

## Ground truth: Riccati-optimal policy

The unperturbed ($\lambda=0$) optimal $\theta^\star$ (`LQ.riccati_optimal`, the two decoupled Riccati recursions) and its value $J^0(\theta^\star)$ — exact, closed-form, no training needed. Every plot below compares the learned policy against this.

In [ ]:
theta_star = env.riccati_optimal(T)
J_star = env.exact_objective(theta_star, 0.0).item()
grad_at_star = env.exact_gradient(theta_star, 0.0)

print("theta* (self gain, population gain):")
print(theta_star)
print(f"J^0(theta*) = {J_star:.6f}")
print(f"||grad J^0(theta*)|| = {grad_at_star.norm().item():.2e}  (should be ~0: theta* is a stationary point)")

mu_star, Sigma_star = env.forward_moments(theta_star, 0.0)
print(f"mu_t under theta*: {mu_star.tolist()}")

## Train (or load cached results)

Loads every saved run under `runs/lq/mid/` for both algorithms; trains first if none exist yet.

In [ ]:
runs_dir = ROOT / "runs" / "lq" / "mid"

runs = []
for alg in cfg.algorithms:
    if not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_continuous("lq", alg, "mid")
    runs += load_runs("lq", alg, "mid")

total_train_seconds = sum(r["elapsed_seconds"] for r in runs)
print(f"{len(runs)} runs loaded; total training compute time: {total_train_seconds:.1f}s ({total_train_seconds / 60:.1f} min)")
for r in sorted(runs, key=lambda r: (r["alg"], r["lam"])):
    print(f"  {r['alg']:<14} lambda={r['lam']:<5} seed={r['seed']}  elapsed={r['elapsed_seconds']:.1f}s  final validation J={r['validation_J'][-1].item():.4f}")

## Evolution of the validation objective

The exact validation objective $J^0(\theta_m)$ (closed-form, not Monte Carlo) every `validate_every` training iterations, one line per $(\text{algorithm},\lambda)$, against the Riccati-optimal reference line. `exact_gradient` should converge smoothly and essentially exactly; `reinforce`'s Monte Carlo noise should be visible.

In [ ]:
fig, ax = viz.plot_validation_curve(runs, optimal_J=J_star)
ax.set_title("Validation objective during training (dashed = Riccati optimum)", loc="left")

## Learned theta vs Riccati-optimal theta

$\theta_t^1$ (self gain) and $\theta_t^2$ (population gain) across $t$, for $\lambda=0.2$'s learned policy under each algorithm, against $\theta^\star$ (dashed).

In [ ]:
by_alg_lambda = {(r["alg"], r["lam"]): r for r in runs}
theta_exact = by_alg_lambda[("exact_gradient", 0.2)]["theta_final"]
theta_reinforce = by_alg_lambda[("reinforce", 0.2)]["theta_final"]

fig, axes = None, None
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
viz.plot_lq_theta(theta_exact, optimal_theta=theta_star, ax=axes[0])
axes[0].set_title("exact_gradient (lambda=0.2)", loc="left")
viz.plot_lq_theta(theta_reinforce, optimal_theta=theta_star, ax=axes[1])
axes[1].set_title("reinforce (lambda=0.2)", loc="left")
fig.tight_layout()

## State distribution over time: learned vs optimal

$\mu_t^\theta\pm\sigma_t^\theta$ (mean $\pm$ 1 std of the Gaussian state law) under the learned policy, against the Riccati-optimal trajectory.

In [ ]:
mu_learned, Sigma_learned = env.forward_moments(theta_exact, 0.0)
fig, ax = viz.plot_gaussian_flow(mu_learned, Sigma_learned, optimal_mu=mu_star, optimal_Sigma=Sigma_star, label="learned (exact_gradient, lambda=0.2)")
ax.set_title("State law X_t: learned vs Riccati-optimal", loc="left")

## $J^\lambda$ vs $J^0$, at the learned and optimal theta

Both sides of this comparison are exact closed-form evaluations (`LQ.exact_objective`), not Monte Carlo estimates — LQ's whole purpose is to remove that source of noise from the comparison. The reference's own $O(\lambda^2)$ bias identity ($J^\lambda(\theta)-J^0(\theta)=\lambda^2\rho^2\mathcal B(\theta)$, `LQ.objective_bias`) is checked exactly in `tests/test_lq.py`.

In [ ]:
lambdas = cfg.lambdas
J_lambda_at_learned = {lam: env.exact_objective(theta_exact, lam).item() for lam in lambdas}
J_lambda_at_optimal = {lam: env.exact_objective(theta_star, lam).item() for lam in lambdas}

fig, ax = new_figure()
ax.plot(lambdas, list(J_lambda_at_learned.values()), color=color_for(0), linewidth=2, marker="o", markersize=6, label="J^lambda(theta_learned)")
ax.plot(lambdas, list(J_lambda_at_optimal.values()), color=color_for(1), linewidth=2, marker="s", markersize=6, label="J^lambda(theta*)")
ax.axhline(J_star, color="black", linestyle="--", linewidth=1.5, label="J^0(theta*)")
apply_style(ax, xlabel="lambda", ylabel="J^lambda")
style_legend(ax)
ax.set_title("Perturbed objective vs lambda, at learned and optimal theta", loc="left")

## Error sensitivity vs time

The exact variance bias $\Sigma_t^{\theta,\lambda}-\Sigma_t^{\theta,0}=\lambda^2\rho^2V_t^\theta$ (the mean is *exactly* unaffected by $\lambda$: `LQ.forward_moments`'s `mu` doesn't depend on `lam` at all) at the learned theta, across $t$, for a few $\lambda$. Shows how the perturbation's effect compounds forward through the horizon.

In [ ]:
_, Sigma_0 = env.forward_moments(theta_exact, 0.0)
variance_bias = {}
for lam in (0.1, 0.2, 0.4, 0.8):
    _, Sigma_lam = env.forward_moments(theta_exact, lam)
    variance_bias[f"lambda={lam}"] = Sigma_lam - Sigma_0

fig, ax = viz.plot_population_fractions(variance_bias)
ax.set_ylim(None, None)  # plot_population_fractions defaults to [0,1] for probabilities; this is a cost/variance quantity
apply_style(ax, xlabel="t", ylabel="Sigma_t^{theta,lambda} - Sigma_t^{theta,0}")
ax.set_title("Perturbation variance bias vs t (theta_learned, exact_gradient)", loc="left")

## Gradient bias/variance: reinforce vs the exact gradient

Unlike every other benchmark, the ground truth here (`LQ.exact_gradient`) is exact, not another estimator — so this isolates `reinforce`'s Monte Carlo bias/variance *exactly*, with zero oracle noise of its own. Summarized as $\|\text{bias}\|$/$\|\text{std}\|$ over the $2T$-dimensional theta (as in the advertising/cybersecurity notebooks' aggregate-norm treatment).

In [ ]:
from mfc.algorithms.lq import reinforce_step

reps = 30
bias_norm, std_norm = {}, {}
gen = torch.Generator(device="cpu").manual_seed(0)
for lam in lambdas:
    theta_lam = by_alg_lambda[("reinforce", lam)]["theta_final"]
    oracle = env.exact_gradient(theta_lam, lam)
    theta_attached = theta_lam.clone().requires_grad_(True)
    samples = torch.stack([reinforce_step(env, theta_attached, lam, cfg.B, generator=gen) for _ in range(reps)])
    bias_norm[lam] = (samples.mean(dim=0) - oracle).norm().item()
    std_norm[lam] = samples.std(dim=0).norm().item()

fig, ax = viz.plot_horizon_scaling(bias_norm, xlabel="lambda", ylabel="norm over theta (T*2 components)", label="||bias||", integer_xaxis=False)
viz.plot_horizon_scaling(std_norm, xlabel="lambda", label="||std||", color_index=1, integer_xaxis=False, ax=ax)
ax.set_title("reinforce gradient estimator vs the exact gradient: bias/std norms vs lambda", loc="left")

## Horizon scaling

Final validation objective and $\|\theta_{\text{learned}}-\theta^\star\|$ vs horizon $T$, using `runs/lq/main/` if it has been populated (`scripts/train.py --env lq --alg <alg> --config main`); otherwise this cell notes that main-tier data is needed and skips.

In [ ]:
main_runs_dir = ROOT / "runs" / "lq" / "main"
main_runs = [r for alg in cfg.algorithms if (main_runs_dir / f"{alg}_T3_lam0.2_seed0.pt").exists() for r in load_runs("lq", alg, "main")] if main_runs_dir.exists() else []

if not main_runs:
    print("no runs/lq/main/ data yet -- run scripts/train.py --env lq --alg exact_gradient --config main "
          "(and --alg reinforce) to populate this cell")
else:
    from configs.lq import MAIN
    final_J_by_T = {}
    theta_err_by_T = {}
    for T_h in MAIN.horizons:
        group = [r for r in main_runs if r["alg"] == "exact_gradient" and r["lam"] == 0.2 and r["T"] == T_h]
        theta_h_star = env.riccati_optimal(T_h)
        final_J_by_T[T_h] = sum(r["validation_J"][-1].item() for r in group) / len(group)
        theta_err_by_T[T_h] = sum((r["theta_final"] - theta_h_star).norm().item() for r in group) / len(group)

    fig, ax = viz.plot_horizon_scaling(final_J_by_T, ylabel="final validation J (exact_gradient, lambda=0.2)", label="J")
    ax.set_title("Horizon scaling: final objective", loc="left")
    fig, ax = viz.plot_horizon_scaling(theta_err_by_T, ylabel="||theta_learned - theta*||", label="theta error", color_index=1)
    ax.set_title("Horizon scaling: theta error vs Riccati optimum", loc="left")

## Generalization without retraining

Evaluating the $\lambda=0.2$ `exact_gradient`-learned $\theta$ exactly (no retraining, no Monte Carlo — `LQ.exact_objective` is closed-form) under a different initial law, a stronger perturbation intensity, and model misspecification. (LQ's $\theta$ is horizon-specific, so unlike the other notebooks there is no "different $T$" scenario here — see `mfc.environments.lq`'s module docstring.)

In [ ]:
scenarios = {
    "baseline (mu0, rho)": env,
    "mu0 x 2": LQ(LQConfig(mu0=env.config.mu0 * 2), device="cpu"),
    "mu0 = 0": LQ(LQConfig(mu0=0.0), device="cpu"),
    "rho x 2 (stronger perturbation)": LQ(LQConfig(rho=env.config.rho * 2), device="cpu"),
    "20% weaker mean-field coupling (c, kappa, kappa_T)": LQ(LQConfig(c=env.config.c * 0.8, kappa=env.config.kappa * 0.8, kappa_T=env.config.kappa_T * 0.8), device="cpu"),
    "20% stronger self-transition (a)": LQ(LQConfig(a=env.config.a * 1.2), device="cpu"),
}
gen_results = [{"name": name, "J": sc_env.exact_objective(theta_exact, 0.2)} for name, sc_env in scenarios.items()]
fig, ax = viz.plot_generalization(gen_results)
ax.set_title("J^0.2 under different scenarios (theta fixed at lambda=0.2's learned value, no retraining)", loc="left")

## Missing mean-field term: exact_gradient vs reinforce

`reinforce` treats the population mean as exogenous — the same ablation `mfc.algorithms.reinforce` implements for the discrete benchmarks (context.md: "Show missing mean-field term by comparison with reinforce"). Final validation objective for both algorithms, across seeds, at $\lambda=0.2$.

In [ ]:
for alg in cfg.algorithms:
    group = [r for r in runs if r["alg"] == alg and r["lam"] == 0.2]
    finals = torch.tensor([r["validation_J"][-1].item() for r in group])
    print(f"{alg:<14} final validation J: mean={finals.mean().item():.4f}  std={finals.std().item() if len(finals) > 1 else 0.0:.4f}  (J*={J_star:.4f})")

## Sample trajectories: learned vs optimal

One sampled $X_t$ trajectory under the learned policy ($\lambda=0.2$, `exact_gradient`) and one under $\theta^\star$, both from $X_0\sim\mathcal N(\mu_0,\Sigma_0)$.

In [ ]:
gen_l = torch.Generator(device="cpu").manual_seed(0)
gen_o = torch.Generator(device="cpu").manual_seed(1)
learned_traj = env.rollout(theta_exact, lam=0.0, B=1, generator=gen_l)["X"][:, 0]
optimal_traj = env.rollout(theta_star, lam=0.0, B=1, generator=gen_o)["X"][:, 0]

fig, ax = viz.plot_trajectories(learned_traj, optimal_traj)
ax.set_ylabel("X_t")
ax.set_title("Sample trajectories (learned solid, optimal dashed)", loc="left")

## Summary

Total notebook runtime (including any training performed in this run):

In [ ]:
print(f"total notebook runtime: {time.perf_counter() - _notebook_start:.1f}s")